In [1]:

import fasttext

# 1. Load the model once
print("Loading model...")
model = fasttext.load_model(r"C:\Users\NITRO V 15\Desktop\QUERYYY\model\QueryClassifier_6Classes.bin")

def build_pipeline_for_query(query: str) -> dict:
    cleaned = query.replace("\n", " ").lower().strip()
    
    # Fast path for short queries (under 5 words)
    if len(cleaned.split()) < 5:
        return {
            "query_type": "retrieval (fast-path)",
            "confidence": 1.0,
            "pipeline": {"retrievers": ["dense", "bm25"], "reranker": False, "expansion": None}
        }
        
    # Run FastText prediction
    labels, probabilities = model.predict(cleaned, k=1)
    predicted_label = labels[0].replace("__label__", "")
    confidence = probabilities[0]
    
    # Fallback if model is unsure (< 60% confidence)
    if confidence < 0.60:
        predicted_label = "retrieval"
        
    # Map predictions to dynamic RAG pipeline settings
    pipeline_configs = {
        "retrieval":        {"retrievers": ["dense", "bm25"], "reranker": False, "expansion": None},
        "comparison":       {"retrievers": ["dense", "bm25"], "reranker": True, "expansion": None},
        "multi_hop":        {"retrievers": ["dense", "bm25", "graphrag"], "reranker": True, "expansion": "multi_query"},
        "summarization":    {"retrievers": ["dense"], "raptor_summaries": True, "reranker": False, "expansion": None},
        "metadata_filter":  {"retrievers": ["dense_filtered"], "self_query_filter": True, "reranker": False, "expansion": None},
        "follow_up":        {"retrievers": ["dense"], "history_memory": True, "reranker": False, "expansion": None}
    }
    
    return {
        "query_type": predicted_label,
        "confidence": float(confidence),
        "pipeline": pipeline_configs.get(predicted_label, pipeline_configs["retrieval"])
    }

# 2. Batch Test Queries
test_queries = {
    "1. RETRIEVAL":        "what is the melting point of gold in degrees celsius",
    "2. COMPARISON":       "compare the features of nextjs vs vite for building react web applications",
    "3. MULTI_HOP":        "Who is the CEO of the company that manufactures the engine oil recommended for 4-wheeler vehicles in low-temperature environments?",
    "4. SUMMARIZATION":    "please write a detailed summary of the entire chapter three of the book",
    "5. METADATA_FILTER":  "find all PDF reports uploaded by the HR department in the year 2025",
    "6. FOLLOW_UP":        "and what was the final decision about that budget increase"
}

print("\n================ RUNNING BATCH PATHWAY TESTS ================")
for expected_path, query in test_queries.items():
    config = build_pipeline_for_query(query)
    print(f"\n[Expected Class]: {expected_path}")
    print(f"  Query:      \"{query}\"")
    print(f"  Classified:  {config['query_type'].upper()} (Confidence: {config['confidence']:.2%})")
    print(f"  Pipeline:    {config['pipeline']}")
    print("-" * 75)

Loading model...

================ RUNNING BATCH PATHWAY TESTS ================

[Expected Class]: 1. RETRIEVAL
  Query:      "what is the melting point of gold in degrees celsius"
  Classified:  RETRIEVAL (Confidence: 97.74%)
  Pipeline:    {'retrievers': ['dense', 'bm25'], 'reranker': False, 'expansion': None}
---------------------------------------------------------------------------

[Expected Class]: 2. COMPARISON
  Query:      "compare the features of nextjs vs vite for building react web applications"
  Classified:  COMPARISON (Confidence: 100.00%)
  Pipeline:    {'retrievers': ['dense', 'bm25'], 'reranker': True, 'expansion': None}
---------------------------------------------------------------------------

[Expected Class]: 3. MULTI_HOP
  Query:      "Who is the CEO of the company that manufactures the engine oil recommended for 4-wheeler vehicles in low-temperature environments?"
  Classified:  MULTI_HOP (Confidence: 77.58%)
  Pipeline:    {'retrievers': ['dense', 'bm25', 'gr